# XTNeighbor Nearest Neighbor Search Benchmark

This notebook aims to provide a reproducible benchmark of XTNeighbor on nearest neighbor search task. That is, given a dataset of Adaptive Immune Receptor sequences and a Levenshtein distance threshold, the algorithm identifies all pairs of sequences that have their similarity within the threshold value. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

The notebook is divided into 6 steps as follow:
1. __Configuration:__ select the number of experiment repeats and maximum dataset size. Note that the largest option of dataset size requires high-RAM VM which requires paid Google Colab account.
2. __Benchmark Setup:__ install dependencies and implementations of various algorithms and compile them if need be.
3. __Algorithm Benchmark:__ perform benchmark on 5 algorithms implemented in Python to measure the speed of each approach at threshold `d=1,2`. They include exhautive search, bk-tree, representation design, combinatorial lookup, and symmetric deletion.
4. __Symmetric Deletion Benchmark:__ perform benchmark on each variation of the implementation. They include Python implementation, XTNeighbor, and XTNeighbor without streaming.
5. __Result Download:__ download the benchmark measurement as csv file.

Warning: some sections take up to 1 hour to run. The run time is remarked at each section's heading.

More information can be found in our [preprint paper](https://doi.org/10.48550/arXiv.2403.09010) and our [Github repository](https://github.com/heartnetkung/XT-neighbor).

## 1. Configuration

In [1]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 1 # @param ["1", "10", "30"] {type:"raw"}
high_ram = False # @param {type:"boolean"}

## 2. Benchmark Setup (run time < 5 min)

install dependency

In [2]:
! pip install -q pyrepseq pybktree symscan

In [ ]:
import os.path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import random
import pybktree
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

compile XTNeighbor without streaming

In [ ]:
! mkdir -p {repo_path}xtneighbor/build
! cd {repo_path}xtneighbor/build; cmake -S .. -B .;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/1.0/build
[100%] Built target xt_neighbor


compile XTNeighbor

In [ ]:
! mkdir -p {repo_path}/xtneighbor_streaming/build
! cd {repo_path}/xtneighbor_streaming/build; cmake -S .. -B .;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/2.0/build
[100%] Built target xt_neighbor


prepare input data

In [6]:
N_FILES=6

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

first row: CASSLDSYEQYF
len: 57,472,488


## 2. Algorithm Benchmark (run time ~ 15 min at n_repeat=1)

exhaustive search implementation

In [7]:
def exhaustive_search(seqs, max_edits):
  ans = []
  for i in range(len(seqs)):
    for j in range(len(seqs)):
        if i >= j:
            continue
        dist = levenshtein_distance(seqs[j], seqs[i], score_cutoff=max_edits)
        if dist <= max_edits:
            ans += [(i, j, dist)]
  return ans

bktree implementation

In [8]:
def build_index(seqs):
    ans = {}
    for index, seq in enumerate(seqs):
        if seq not in ans:
            ans[seq] = []
        ans[seq].append(index)
    return ans


def bktree(seqs, max_edits=1):
    ans = []
    index = build_index(seqs)

    tree = pybktree.BKTree(levenshtein_distance, np.unique(seqs))
    for x_index, x_seq in enumerate(seqs):
        for edit_distance, y_seq in tree.find(x_seq, max_edits):
            for y_index in index[y_seq]:
                if x_index != y_index:
                    ans.append((x_index, y_index, edit_distance))
    return ans

benchmarking code

In [9]:
# 100 is the warm up
sizes = [100, 1_000, 3_000, 10_000, 30_000, 100_000]
algorithms = {
    'kdtree':pyrepseq.kdtree,
    'symdel':pyrepseq.symdel,
    'combinatorial_lookup': pyrepseq.hash_based,
    'exhaustive_search': exhaustive_search,
    'bktree': bktree}
limits = {
    'exhaustive_search_1':100_000,
    'exhaustive_search_2':100_000,
    'combinatorial_lookup_2':10_000,
    'kdtree_2':100_000,
    'exhaustive_search_2':100_000,
    'bktree_2': 100_000}
alg_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

def run_alg_exp(distance):
    for i in range(n_repeat):
        for size in sizes:
            subset = random.Random(i).sample(data,size)
            for alg_name in algorithms:
                limit = limits.get(f"{alg_name}_{distance}")
                if limit is not None and limit <=size:
                    continue
                # perform
                start = time.time()
                algorithms[alg_name](subset,distance)
                end = time.time()

                # record
                print(f'{size:,}',alg_name,i,round((end-start)*100)/100)
                alg_result['runtime'].append(end-start)
                alg_result['algorithm'].append(alg_name)
                alg_result['input_size'].append(size)
                alg_result['distance'].append(distance)

In [10]:
run_alg_exp(distance=1)

100 kdtree 0 0.0
100 symdel 0 0.0
100 combinatorial_lookup 0 0.01
100 exhaustive_search 0 0.0
100 bktree 0 0.0
1,000 kdtree 0 0.4
1,000 symdel 0 0.01
1,000 combinatorial_lookup 0 0.09
1,000 exhaustive_search 0 0.06
1,000 bktree 0 0.03
3,000 kdtree 0 0.16
3,000 symdel 0 0.01
3,000 combinatorial_lookup 0 0.26
3,000 exhaustive_search 0 0.55
3,000 bktree 0 0.17
10,000 kdtree 0 1.31
10,000 symdel 0 1.0
10,000 combinatorial_lookup 0 0.92
10,000 exhaustive_search 0 6.87
10,000 bktree 0 1.23
30,000 kdtree 0 8.76
30,000 symdel 0 1.55
30,000 combinatorial_lookup 0 2.7


KeyboardInterrupt: 

In [ ]:
run_alg_exp(distance=2)

100 kdtree 0 0.0
100 symspell 0 0.0
100 combinatorial_lookup 0 6.64
100 exhaustive_search 0 0.0
100 bktree 0 0.0
1,000 kdtree 0 0.05
1,000 symspell 0 0.04
1,000 combinatorial_lookup 0 64.41
1,000 exhaustive_search 0 0.07
1,000 bktree 0 0.1
3,000 kdtree 0 0.41
3,000 symspell 0 1.6
3,000 combinatorial_lookup 0 196.23
3,000 exhaustive_search 0 0.61
3,000 bktree 0 0.69
10,000 kdtree 0 4.52
10,000 symspell 0 3.01
10,000 exhaustive_search 0 7.6
10,000 bktree 0 6.43
30,000 kdtree 0 47.1
30,000 symspell 0 5.97
30,000 exhaustive_search 0 86.89
30,000 bktree 0 57.09
100,000 symspell 0 15.77


## 3. Symmetric Deletion  Implementation Benchmark (run time ~ 10 min at n_repeat=1)

check GPU availability

In [ ]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

standardize all algorithms to the same API

In [ ]:
def prepare(seqs):
  with open("input1.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in seqs)
  with open("input2.txt","w") as file2:
    file2.writelines(seq+'\n' for seq in (['cdr3']+seqs))

def xt_neighbor_1(seqs,threshold,_len,verbose=False): #verbose is ignored
  ! {repo_path}xtneighbor/build/xt_neighbor -p "input1.txt" -n "$_len" -d "$threshold"

def xt_neighbor_2(seqs,threshold,_len,verbose=False):
  if verbose:
    ! {repo_path}xtneighbor_streaming/build/xt_neighbor -i "input2.txt" -n "$_len" -d "$threshold" -V
  else:
    ! {repo_path}xtneighbor_streaming/build/xt_neighbor -i "input2.txt" -n "$_len" -d "$threshold"

def symdel(seqs,threshold,_len,verbose=False): #verbose is ignored
  return pyrepseq.symdel(seqs,max_edits=threshold)

def run_symscan(seqs,threshold,_len,verbose=False): #verbose is ignored
  return symscan.get_neighbors_within(seqs, max_distance=threshold)



benchmarking code

In [ ]:
# 100 is the warm up
sizes = [100, 10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000, 10_000_000, 30_000_000]

limits = {
    'symscan_1':30_000_000,
    'symscan_2':10_000_000,
    'V1_1':10_000_000,
    'V1_2':1_000_000,
    'symdel_1':10_000_000,
    'symdel_2':3_000_000,
    'V2_1':30_000_000,
    'V2_2':10_000_000}

algorithms = {
    'V2':xt_neighbor_2,
    'V1':xt_neighbor_1,
    'symdel':symdel,
    'symscan':run_symscan,
}
verbose = False

symdel_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

def run_exp(distance):
    for i in range(n_repeat):
        for size in sizes:
            subset = random.Random(i).sample(data,size)
            prepare(subset)
            for alg_name in algorithms:
                limit = limits.get(f"{alg_name}_{distance}")
                if limit is not None and limit <size:
                    continue

                # perform
                start = time.time()
                algorithms[alg_name](subset,distance,size,verbose)
                end = time.time()

                # record
                print(f'{size:,}',alg_name,i,round((end-start)*100)/100)
                symdel_result['runtime'].append(end-start)
                symdel_result['algorithm'].append(alg_name)
                symdel_result['input_size'].append(size)
                symdel_result['distance'].append(distance)

In [ ]:
run_exp(distance=1)

total output length: 0
100 V2 0 0.56
Success! Number of triplet: 0
100 V1 0 0.42
100 symdel 0 0.0
100 symscan 0 0.0
total output length: 19
10,000 V2 0 0.52
Success! Number of triplet: 19
10,000 V1 0 0.44
10,000 symdel 0 0.08
10,000 symscan 0 0.0
total output length: 152
30,000 V2 0 0.54
Success! Number of triplet: 152
30,000 V1 0 0.44
30,000 symdel 0 0.79
30,000 symscan 0 0.01
total output length: 1,581
100,000 V2 0 0.57
Success! Number of triplet: 1,581
100,000 V1 0 0.44
100,000 symdel 0 4.2
100,000 symscan 0 0.02
total output length: 14,260
300,000 V2 0 0.72
Success! Number of triplet: 14,260
300,000 V1 0 0.47
300,000 symdel 0 9.02
300,000 symscan 0 0.08
total output length: 159,453
1,000,000 V2 0 1.07
Success! Number of triplet: 159,453
1,000,000 V1 0 0.56
1,000,000 symdel 0 23.84
1,000,000 symscan 0 0.28
total output length: 1,438,868
3,000,000 V2 0 2.02
Success! Number of triplet: 1,438,868
3,000,000 V1 0 0.84
3,000,000 symdel 0 55.26
3,000,000 symscan 0 0.95
total output length:

: 

In [ ]:
run_exp(distance=2)

total output length: 0
100 V2 0 0.56
Success! Number of triplet: 0
100 V1 0 0.44
100 symdel 0 0.02
100 symscan 0 0.0
total output length: 611
10,000 V2 0 0.54
Success! Number of triplet: 611
10,000 V1 0 0.45
10,000 symdel 0 0.58
10,000 symscan 0 0.01
total output length: 5,325
30,000 V2 0 0.6
Success! Number of triplet: 5,325
30,000 V1 0 0.45
30,000 symdel 0 1.97
30,000 symscan 0 0.04
total output length: 57,380
100,000 V2 0 0.77
Success! Number of triplet: 57,380
100,000 V1 0 0.47
100,000 symdel 0 6.99
100,000 symscan 0 0.15
total output length: 512,106
300,000 V2 0 1.32
Success! Number of triplet: 512,106
300,000 V1 0 0.53
300,000 symdel 0 31.01
300,000 symscan 0 0.54
total output length: 5,718,583
1,000,000 V2 0 3.18
Success! Number of triplet: 5,718,583
1,000,000 V1 0 0.78
1,000,000 symdel 0 128.86
1,000,000 symscan 0 2.47
total output length: 51,467,657
3,000,000 V2 0 9.91


## 4. Result Download (run time < 1 min)

In [ ]:
pd.DataFrame(alg_result).to_csv('cpu_benchmark.csv')
if colab:
    files.download('cpu_benchmark.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pd.DataFrame(symdel_result).to_csv('gpu_benchmark.csv')
if colab:
    files.download('gpu_benchmark.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>